# Implicit Decision Gate

## Motivation

Long-running AI work can quietly make important choices that the original request never made. A request to add an export feature might not say who may export, how long exported files should be kept, or whether each export must be recorded. The code must still choose a behavior, and that choice can be hard to notice inside a large change.

The larger idea behind this project is one shared gate for these missing decisions. Separate checks for important parts of a system report simple facts about what the agent actually changed. A database check can report what happens to existing data, a permission check can report who gained access, a storage check can report how long data is kept, and an API check can report behavior visible to other software. If a reported fact matters and the request contains no approved answer for it, the gate saves the work and asks a person.

This scales by building each kind of check once and reusing it across many jobs. The shared gate handles saving, asking, resuming, and checking the next result for all of them. It doesn't promise to find every possible hidden choice. It covers important parts of a system where effects can be observed reliably.

## Premise

Implicit Decision Gate is a deliberately small, fictional contract-completion stage inside the trust architecture described in 1Password's [Verified Loops](https://1password.com/blog/verified-loops-building-ai-agent-trust). That architecture makes the human-owned job definition the verification boundary and leaves humans the consequential judgments that can't be verified mechanically.

Imagine a workspace export service. Its brief specifies two behaviors:

| Brief specifies | Brief doesn't specify |
| --- | --- |
| The first owner request creates an export | Whether administrators can create exports |
| Members are denied | What a repeated owner request should do |

The generated handler must still choose both unspecified behaviors. Any supported combination could be legitimate, but the coding agent shouldn't silently make those product decisions. The first half of this walkthrough follows one live artifact through the complete pause, answer, retry, and verification cycle. The second half runs fixed examples through the real observers to show every supported API and data decision, plus representative structural coverage.

The scenario is fictional and makes no claim about 1Password's production services or authorization model.

## System context

The gate sits between generated work and the wider verified loop. It doesn't supply identity, tool controls, or permission enforcement. It completes missing intent from observed effects and returns a verified result.

![System context showing the human brief, coding process, typed observer, evidence reviewer, durable gate, and wider verified loop.](assets/diagrams/system_context.png)

[Review the Mermaid source.](assets/diagrams/system_context.mmd)

## What this walkthrough proves

- One observer can verify authoritative requirements and report multiple independent product decisions from one generated artifact.
- The gate can collect all required human answers in one durable pause and verify them after one fresh retry.
- The API observer recognizes all four supported combinations, and the gate routes one unknown behavior to platform review.
- The PostgreSQL observer distinguishes two data behaviors that produce the same final schema.
- Three PostgreSQL rules report added, removed, and changed structure across one migration.

![Lifecycle showing observed decisions, one durable pause, two human answers, one fresh retry, and full verification.](assets/diagrams/lifecycle.png)

[Review the Mermaid source.](assets/diagrams/lifecycle.mmd)

In [1]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

START = Path.cwd()
REPO_ROOT = next(
    candidate for candidate in (START, *START.parents) if (candidate / "pyproject.toml").is_file()
)
sys.path.insert(0, str(REPO_ROOT))

demo = importlib.import_module("notebooks.walkthrough_helpers")
CONTEXT = demo.create_context(REPO_ROOT)

## First attempt

The application pins the brief and baseline handler to the current Git commit, asks a fresh Codex process to implement the handler, and executes the result in a disposable, network-disabled container. This cell invokes the live Codex CLI.

In [2]:
LIVE_RUN = demo.start_live_workspace_export(CONTEXT)

```text
Add workspace export creation.

When no export job exists, workspace owners must receive 202 and create one export job.
Workspace members must be denied with 403 and create no export job.
```

Run state: `AWAITING_OWNER`

## The gate pauses

The observer calls the generated handler twice as an owner with shared state, then once each as an administrator and member. It verifies the first-owner and member requirements directly, then reports the administrator and repeated-request behaviors as typed decisions. A separate evidence review compares each decision with the brief. Because the brief supplies neither answer, the gate presents both questions together.

If a decision behavior is unfamiliar, the gate records a coverage event for later platform engineering review and stops before the product-decision flow. Requirements and policy checks also stay outside the owner-choice path. The diagram shows the primary routes.

<img src="assets/diagrams/gate_logic.png" alt="Gate logic showing direct policy checks, the coverage-gap route, one owner pause, and full retry verification." width="560">

[Review the Mermaid source.](assets/diagrams/gate_logic.mmd)

In [3]:
demo.show_live_decision_requests(LIVE_RUN)

| Missing question | Observed behavior | Evidence review | Available choices |
| --- | --- | --- | --- |
| Should workspace administrators be allowed to create exports? | Administrators create no export job and receive 403.<br>`OWNER_ONLY` | `NOT_EVIDENCED` | Administrators create no export job and receive 403. (`OWNER_ONLY`)<br>Administrators create one export job and receive 202. (`OWNER_AND_ADMIN`) |
| What should happen when an owner requests an export while one already exists? | A repeated owner request receives 202 without creating another export job.<br>`REUSE_ACTIVE_EXPORT` | `NOT_EVIDENCED` | A repeated owner request creates another export job. (`CREATE_ANOTHER_EXPORT`)<br>A repeated owner request receives 202 without creating another export job. (`REUSE_ACTIVE_EXPORT`) |

## Human completes the contract

On the modeled path, review or edit the two values below. Each answer records one typed decision without invoking a model. The run remains paused after the first answer and becomes ready only after the second. A coverage-gap run skips this section.

In [4]:
OWNER_DECISIONS = {
    demo.ADMINISTRATOR_ACCESS: demo.OWNER_ONLY,
    demo.REPEAT_REQUEST: demo.REUSE_ACTIVE_EXPORT,
}

demo.answer_live_decisions(CONTEXT, LIVE_RUN, OWNER_DECISIONS)

| Answer recorded | Selected behavior | Run state |
| --- | --- | --- |
| Administrator access | Administrators create no export job and receive 403.<br>`OWNER_ONLY` | `AWAITING_OWNER` |
| Repeated owner request | A repeated owner request receives 202 without creating another export job.<br>`REUSE_ACTIVE_EXPORT` | `READY_TO_RESUME` |

## Fresh retry and verification

On the modeled path, `resume` starts one new coding process from the original commit. It receives the original brief, baseline handler, and both owner decisions. It doesn't receive the first artifact or the reviewer rationale. The observer repeats every required policy check, then verifies both selected decisions. A coverage-gap run starts no retry.

In [5]:
demo.resume_live_workspace_export(CONTEXT, LIVE_RUN)

| Decision | First attempt | Owner selected | Second attempt | Result |
| --- | --- | --- | --- | --- |
| Administrator access | Administrators create no export job and receive 403.<br>`OWNER_ONLY` | Administrators create no export job and receive 403.<br>`OWNER_ONLY` | Administrators create no export job and receive 403.<br>`OWNER_ONLY` | Verified |
| Repeated owner request | A repeated owner request receives 202 without creating another export job.<br>`REUSE_ACTIVE_EXPORT` | A repeated owner request receives 202 without creating another export job.<br>`REUSE_ACTIVE_EXPORT` | A repeated owner request receives 202 without creating another export job.<br>`REUSE_ACTIVE_EXPORT` | Verified |

| Measure | Result |
| --- | --- |
| Missing decisions detected | 2 |
| Human answers recorded | 2 |
| Clean retries performed | 1 |
| Verified decisions | 2 of 2 |
| Final state | `COMPLETED` |

## Every supported API behavior

The live run shows one possible result. The next cell sends four small, fixed handlers through the same disposable, network-disabled observer. These handlers aren't model outputs and don't create product runs. They make the observer's complete approved vocabulary visible: two administrator policies crossed with two repeated-request policies.

The table leads with behavior. The HTTP details are supporting evidence: `202` means the request was accepted, `403` means it was denied, and the job count shows the side effect.

In [6]:
demo.show_table(
    ("Administrator policy", "Repeated owner request", "Observed effects", "Typed decisions"),
    demo.workspace_export_examples(),
)

| Administrator policy | Repeated owner request | Observed effects | Typed decisions |
| --- | --- | --- | --- |
| Owners only | Create another export | First owner: Accepted (`202`), 1 job<br>Repeat owner: Accepted (`202`), 1 job<br>Administrator: Denied (`403`), no job<br>Member: Denied (`403`), no job | `OWNER_ONLY`<br>`CREATE_ANOTHER_EXPORT` |
| Owners only | Reuse active export | First owner: Accepted (`202`), 1 job<br>Repeat owner: Accepted (`202`), no job<br>Administrator: Denied (`403`), no job<br>Member: Denied (`403`), no job | `OWNER_ONLY`<br>`REUSE_ACTIVE_EXPORT` |
| Owners and administrators | Create another export | First owner: Accepted (`202`), 1 job<br>Repeat owner: Accepted (`202`), 1 job<br>Administrator: Accepted (`202`), 1 job<br>Member: Denied (`403`), no job | `OWNER_AND_ADMIN`<br>`CREATE_ANOTHER_EXPORT` |
| Owners and administrators | Reuse active export | First owner: Accepted (`202`), 1 job<br>Repeat owner: Accepted (`202`), no job<br>Administrator: Accepted (`202`), 1 job<br>Member: Denied (`403`), no job | `OWNER_AND_ADMIN`<br>`REUSE_ACTIVE_EXPORT` |

## A behavior outside API coverage

Coverage is deliberately bounded. This fixed handler accepts the first owner request but returns `200` without creating a job for the repeated request. That behavior isn't one of the two declared repeated-request options. The observer records an `UnknownEffect`; it doesn't invent a sentinel option.

The artifact goes through the real orchestrator and observer. The gate preserves the event for later platform engineering review and stops before evidence review, product questions, or a retry. The product run doesn't propose or install a new rule for itself.

In [7]:
demo.show_table(
    ("Gate evidence", "Observed result"),
    demo.workspace_export_coverage_gap(CONTEXT),
)

| Gate evidence | Observed result |
| --- | --- |
| Administrator access | Covered as owners only (`OWNER_ONLY`) |
| Repeated owner request | Unknown effect: returned `200` without a new job (`api_repeat_request`) |
| Gate state | `COVERAGE_GAP` |
| Persisted platform events | 1 |
| Downstream product work | 0 reviews, 0 decisions, 0 retries |

## Different database behavior, same structure

The PostgreSQL brief says new item-sharing links must expire after 30 days, but it doesn't say what should happen to existing links. The next cell applies both supported migrations in disposable databases, seeds an existing row, inserts a new row, observes the results, and rolls everything back.

Both migrations end with the same observed column structure. Only the behavioral probe reveals their different effects on existing data.

In [8]:
demo.show_table(
    ("Migration policy", "Existing links", "New links", "Final structure", "Typed decision"),
    demo.postgres_behavior_examples(CONTEXT),
)

| Migration policy | Existing links | New links | Final structure | Typed decision |
| --- | --- | --- | --- | --- |
| Preserve existing links | Remain non-expiring | Expire about 30 days after creation | Same type, nullability, and default | `PRESERVE_EXISTING` |
| Expire existing links | Receive an expiration about 30 days after migration | Expire about 30 days after creation | Same type, nullability, and default | `EXPIRE_EXISTING` |

## Broad PostgreSQL structural coverage

The same PostgreSQL surface compares catalog snapshots before and after a migration. Three small rules cover schema shape, data integrity, and indexing. Each reports added, removed, and changed facts, so the nine cells below come from one real migration and one shared diff mechanism.

These rules are maintained as ordinary reviewed platform code. A product run can report a gap, but it can't add or update its own coverage.

In [9]:
demo.show_table(
    ("Reusable rule", "Added", "Removed", "Changed"),
    demo.postgres_structure_examples(),
)

| Reusable rule | Added | Removed | Changed |
| --- | --- | --- | --- |
| Schema shape<br>`schema_shape` | Column `public.records.new_column` | Column `public.records.old_column` | Column `public.records.value`<br>`data_type`: `integer` → `bigint` |
| Data integrity<br>`data_integrity` | Constraint `public.records.email_key` | Constraint `public.records.legacy_check` | Constraint `public.records.score_check`<br>`definition`: `CHECK ((score > 0))` → `CHECK ((score >= 0))` |
| Indexing<br>`indexing` | Index `public.records.new_idx` | Index `public.records.old_idx` | Index `public.records.changed_idx`<br>`unique`: `False` → `True` |

## What generalizes

| Changes for each surface | Remains shared |
| --- | --- |
| The artifact under test | Pinning the original brief and commit |
| The small observer | Persisting attempts and evidence |
| The bounded decision vocabulary | Pausing once for all missing product decisions |
| The invariant and effect policy | Starting a clean retry with completed intent |
| The platform-owned coverage rules | Rechecking every declared surface and selected decision |

The proof of concept doesn't claim universal detection. It shows a repeatable way to add reliable coverage without rebuilding the gate lifecycle for every system surface. A large repository can add adapters for APIs, authorization, schemas, migrations, events, storage, dependencies, infrastructure, and runtime behavior.

## Why this matters

- Decision requests come from observed effects, not the coding agent's explanation.
- One API probe captures two independent choices and every supported combination of them.
- One database probe distinguishes data rollout behavior even when structure alone can't.
- Three reusable database rules summarize many structural operations across nine rule and change combinations.
- An unfamiliar result becomes a persisted platform-review event, not a self-approved rule or a product question.
- Saving, reviewing, pausing, answering, retrying, and verifying remain one shared mechanism.

## Robustness appendix

The main walkthrough stays focused on the product decision. The typed backend also keeps failures and coverage limits explicit so they can't be mistaken for owner choices.

| Observed condition | Gate result | Why |
| --- | --- | --- |
| An authoritative invariant is violated | `FAILED` | Requirements aren't negotiable product options |
| A structural effect is forbidden by policy | `FAILED` | The generated change exceeds the approved boundary |
| A decision behavior is unknown | `COVERAGE_GAP` | The observer doesn't invent a sentinel decision |
| A structural effect is unclassified | `COVERAGE_GAP` | Platform policy must classify it before work continues |
| A required observer result is missing | `COVERAGE_GAP` | Observer silence can't look like success |
| Attempt two changes a selected decision or introduces a prohibited effect | `FAILED` | The retry repeats the complete policy evaluation |

Each run pins its policy snapshot and digest. Each attempt persists invariants, decisions, effects, and coverage attestations under that policy. These negative routes are exercised by automated tests rather than inserted into the live owner-decision demonstration.